# Phase 1 (Unified): One-Hand + Two-Hand Gesture Model

This notebook builds one unified model with fixed **126 features**:
- 2 hands x 21 landmarks x 3 coordinates
- If only 1 hand is detected, the missing hand is zero-padded (63 zeros)
- If no hand is detected, sample is skipped

In [ ]:
# Run once if needed
%pip install mediapipe coremltools scikit-learn opencv-python pandas numpy pyyaml

In [1]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import yaml
import mediapipe as mp

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import coremltools as ct
from coremltools.converters import sklearn as ct_sklearn

DATASET_ROOT = Path('/Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4')
OUTPUT_DIR = DATASET_ROOT / 'phase1_outputs_unified'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATASET_ROOT / 'data.yaml', 'r') as f:
    data_cfg = yaml.safe_load(f)

CLASS_NAMES = data_cfg['names']
INDEX_TO_CLASS = {i: name for i, name in enumerate(CLASS_NAMES)}

print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')
print(f'Output directory: {OUTPUT_DIR}')

XGBoost version 3.2.0 has not been tested with coremltools. You may run into unexpected errors. XGBoost 1.4.2 is the most recent version that has been tested.

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/farrellhrs/miniconda3/envs/cv_env/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/farrellhrs/miniconda3/envs/cv_env/lib/python3.11/site-packages/traitlets/config/application.py",

Classes (12): ['bird', 'boar', 'dog', 'dragon', 'hare', 'horse', 'monkey', 'ox', 'ram', 'rat', 'snake', 'tiger']
Output directory: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/phase1_outputs_unified


In [2]:
def parse_yolo_label_file(label_path):
    rows = []
    with open(label_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            cls_id = int(float(parts[0]))
            bbox = list(map(float, parts[1:5]))
            rows.append((cls_id, bbox))
    return rows


def yolo_to_xyxy(bbox, img_w, img_h):
    x_c, y_c, w, h = bbox
    x1 = int((x_c - w / 2.0) * img_w)
    y1 = int((y_c - h / 2.0) * img_h)
    x2 = int((x_c + w / 2.0) * img_w)
    y2 = int((y_c + h / 2.0) * img_h)

    x1 = max(0, min(x1, img_w - 1))
    y1 = max(0, min(y1, img_h - 1))
    x2 = max(0, min(x2, img_w - 1))
    y2 = max(0, min(y2, img_h - 1))
    return x1, y1, x2, y2


def normalize_landmarks(landmarks_xyz):
    lm = np.array(landmarks_xyz, dtype=np.float32).reshape(21, 3)
    wrist = lm[0].copy()
    lm = lm - wrist

    scale = np.max(np.linalg.norm(lm[:, :2], axis=1))
    if scale < 1e-6:
        scale = 1.0
    lm = lm / scale
    return lm.flatten()


def collect_image_label_pairs(dataset_root):
    pairs = []
    for split in ['train', 'valid', 'test']:
        images_dir = dataset_root / split / 'images'
        labels_dir = dataset_root / split / 'labels'
        if not images_dir.exists() or not labels_dir.exists():
            continue

        for img_path in sorted(images_dir.glob('*')):
            if img_path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
                continue
            label_path = labels_dir / f'{img_path.stem}.txt'
            if label_path.exists():
                pairs.append((split, img_path, label_path))
    return pairs


def assign_hands_to_left_right(result, crop_w):
    left_feat = np.zeros(63, dtype=np.float32)
    right_feat = np.zeros(63, dtype=np.float32)

    if not result.multi_hand_landmarks:
        return left_feat, right_feat, 0

    handedness_list = result.multi_handedness or []
    fallback = []

    for idx, hand_lm in enumerate(result.multi_hand_landmarks):
        pts = [(lm.x, lm.y, lm.z) for lm in hand_lm.landmark]
        feat = normalize_landmarks(pts)

        hand_label = None
        if idx < len(handedness_list):
            hand_label = handedness_list[idx].classification[0].label

        if hand_label == 'Left':
            left_feat = feat
        elif hand_label == 'Right':
            right_feat = feat
        else:
            x_center = np.mean([lm.x for lm in hand_lm.landmark]) * crop_w
            fallback.append((x_center, feat))

    for _, feat in sorted(fallback, key=lambda t: t[0]):
        if np.all(left_feat == 0):
            left_feat = feat
        elif np.all(right_feat == 0):
            right_feat = feat

    detected_count = len(result.multi_hand_landmarks)
    return left_feat, right_feat, detected_count


def extract_unified_126_features(crop_bgr, hands):
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    result = hands.process(crop_rgb)

    if not result.multi_hand_landmarks:
        return None, 0

    crop_h, crop_w = crop_bgr.shape[:2]
    left_feat, right_feat, detected_count = assign_hands_to_left_right(result, crop_w)
    merged = np.concatenate([left_feat, right_feat], axis=0).astype(np.float32)

    if merged.shape[0] != 126:
        raise RuntimeError(f'Unexpected unified feature shape: {merged.shape}')

    return merged, detected_count

In [3]:
all_pairs = collect_image_label_pairs(DATASET_ROOT)
print(f'Total image-label pairs found: {len(all_pairs)}')

X = []
y = []
records = []

skipped_images = 0
skipped_boxes = 0
skipped_no_hands = 0
one_hand_samples = 0
two_hand_samples = 0
usable_samples = 0

mp_hands = mp.solutions.hands
with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.3,
    model_complexity=1,
) as hands:
    for split, img_path, label_path in all_pairs:
        img = cv2.imread(str(img_path))
        if img is None:
            skipped_images += 1
            continue

        img_h, img_w = img.shape[:2]
        labels = parse_yolo_label_file(label_path)
        if not labels:
            skipped_images += 1
            continue

        image_used = False
        for cls_id, bbox in labels:
            x1, y1, x2, y2 = yolo_to_xyxy(bbox, img_w, img_h)
            if x2 <= x1 or y2 <= y1:
                skipped_boxes += 1
                continue

            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                skipped_boxes += 1
                continue

            feat, detected_count = extract_unified_126_features(crop, hands)
            if feat is None:
                skipped_no_hands += 1
                continue

            if detected_count >= 2:
                two_hand_samples += 1
            elif detected_count == 1:
                one_hand_samples += 1

            label_name = INDEX_TO_CLASS.get(cls_id, str(cls_id))
            X.append(feat)
            y.append(label_name)
            records.append({
                'split': split,
                'image_path': str(img_path),
                'label_path': str(label_path),
                'class_id': cls_id,
                'class_name': label_name,
                'bbox_x1': x1,
                'bbox_y1': y1,
                'bbox_x2': x2,
                'bbox_y2': y2,
                'detected_hand_count': int(detected_count),
            })
            usable_samples += 1
            image_used = True

        if not image_used:
            skipped_images += 1

X = np.array(X, dtype=np.float32)
y = np.array(y)

print('=== Unified Extraction Summary ===')
print(f'Usable samples: {usable_samples}')
print(f'One-hand samples: {one_hand_samples}')
print(f'Two-hand samples: {two_hand_samples}')
print(f'Skipped images: {skipped_images}')
print(f'Skipped boxes: {skipped_boxes}')
print(f'Skipped no hands: {skipped_no_hands}')
print(f'Feature matrix shape: {X.shape}')

Total image-label pairs found: 6142


I0000 00:00:1774401119.583775  120183 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774401119.588871  120447 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774401119.594978  120447 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/Users/farrellhrs/miniconda3/envs/cv_env/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


=== Unified Extraction Summary ===
Usable samples: 3961
One-hand samples: 3289
Two-hand samples: 672
Skipped images: 2181
Skipped boxes: 0
Skipped no hands: 2177
Feature matrix shape: (3961, 126)


In [4]:
if len(X) == 0:
    raise RuntimeError('No usable samples were extracted. Check MediaPipe settings and dataset quality.')

feature_cols = [f'f_{i:03d}' for i in range(X.shape[1])]
df_features = pd.DataFrame(X, columns=feature_cols)
df_features['label'] = y

class_distribution = pd.Series(y).value_counts().sort_index()
print('=== Class Distribution ===')
print(class_distribution)

csv_path = OUTPUT_DIR / 'hand_landmarks_dataset_unified.csv'
X_npy_path = OUTPUT_DIR / 'X_landmarks_unified.npy'
y_npy_path = OUTPUT_DIR / 'y_labels_unified.npy'
meta_path = OUTPUT_DIR / 'sample_metadata_unified.csv'
dist_path = OUTPUT_DIR / 'class_distribution_unified.csv'

df_features.to_csv(csv_path, index=False)
np.save(X_npy_path, X)
np.save(y_npy_path, y)
pd.DataFrame(records).to_csv(meta_path, index=False)
class_distribution.to_csv(dist_path, header=['count'])

print(f'Saved dataset CSV: {csv_path}')
print(f'Saved NumPy X: {X_npy_path}')
print(f'Saved NumPy y: {y_npy_path}')
print(f'Saved metadata: {meta_path}')
print(f'Saved class distribution: {dist_path}')

=== Class Distribution ===
bird      124
boar      105
dog       368
dragon    367
hare      118
horse     455
monkey    562
ox        302
ram       370
rat       259
snake     280
tiger     651
Name: count, dtype: int64
Saved dataset CSV: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/phase1_outputs_unified/hand_landmarks_dataset_unified.csv
Saved NumPy X: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/phase1_outputs_unified/X_landmarks_unified.npy
Saved NumPy y: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/phase1_outputs_unified/y_labels_unified.npy
Saved metadata: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/phase1_outputs_unified/sample_metadata_unified.csv
Saved class distribution: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/phase1_outputs_unified/class_distribution_unified.csv


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample',
)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)

print(f'Unified model accuracy: {acc:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred))

cm_df = pd.DataFrame(cm, index=rf_model.classes_, columns=rf_model.classes_)
cm_path = OUTPUT_DIR / 'confusion_matrix_unified.csv'
cm_df.to_csv(cm_path)
print(f'Confusion matrix saved to: {cm_path}')

cm_df

Unified model accuracy: 0.9685

Classification report:
              precision    recall  f1-score   support

        bird       1.00      0.88      0.94        25
        boar       0.91      1.00      0.95        21
         dog       0.99      0.99      0.99        74
      dragon       0.93      0.96      0.95        73
        hare       1.00      0.92      0.96        24
       horse       1.00      1.00      1.00        91
      monkey       0.98      0.98      0.98       113
          ox       0.98      0.93      0.96        60
         ram       0.99      0.95      0.97        74
         rat       1.00      0.98      0.99        52
       snake       0.92      0.96      0.94        56
       tiger       0.94      0.98      0.96       130

    accuracy                           0.97       793
   macro avg       0.97      0.96      0.96       793
weighted avg       0.97      0.97      0.97       793

Confusion matrix saved to: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_

,bird,boar,dog,dragon,hare,horse,monkey,ox,ram,rat,snake,tiger
bird,22,0,0,1,0,0,0,0,0,0,0,2
boar,0,21,0,0,0,0,0,0,0,0,0,0
dog,0,1,73,0,0,0,0,0,0,0,0,0
dragon,0,0,0,70,0,0,0,0,0,0,2,1
hare,0,0,0,1,22,0,0,1,0,0,0,0
horse,0,0,0,0,0,91,0,0,0,0,0,0
monkey,0,0,0,2,0,0,111,0,0,0,0,0
ox,0,0,0,0,0,0,2,56,0,0,1,1
ram,0,1,0,0,0,0,0,0,70,0,0,3
rat,0,0,0,0,0,0,0,0,1,51,0,0


In [6]:
coreml_path = OUTPUT_DIR / 'hand_gesture_rf_unified.mlmodel'
coreml_exported = False
coreml_error = None

input_features = [f'f_{i:03d}' for i in range(X.shape[1])]

try:
    coreml_model = ct_sklearn.convert(
        rf_model,
        input_features=input_features,
        output_feature_names=['classLabel', 'classProbability'],
    )
    coreml_model.short_description = 'Unified one/two-hand gesture classifier from MediaPipe landmarks'
    coreml_model.author = 'CBL Unified Phase 1'
    coreml_model.license = 'For research/education use'
    coreml_model.save(str(coreml_path))
    coreml_exported = True
    print(f'CoreML model exported to: {coreml_path}')
except Exception as e:
    coreml_error = str(e)
    print('CoreML export failed:', coreml_error)

CoreML model exported to: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/phase1_outputs_unified/hand_gesture_rf_unified.mlmodel


In [7]:
summary = {
    'total_samples': int(len(X)),
    'one_hand_samples': int(one_hand_samples),
    'two_hand_samples': int(two_hand_samples),
    'skipped_images': int(skipped_images),
    'skipped_boxes': int(skipped_boxes),
    'skipped_no_hands': int(skipped_no_hands),
    'feature_dim': int(X.shape[1]),
    'model_accuracy': float(acc),
    'coreml_exported': bool(coreml_exported),
    'coreml_error': coreml_error,
    'dataset_csv': str(csv_path),
    'coreml_model': str(coreml_path),
}

summary_path = OUTPUT_DIR / 'training_summary_unified.csv'
pd.DataFrame([summary]).to_csv(summary_path, index=False)

print('=== Unified Pipeline Summary ===')
for k, v in summary.items():
    print(f'{k}: {v}')
print(f'Summary saved to: {summary_path}')

=== Unified Pipeline Summary ===
total_samples: 3961
one_hand_samples: 3289
two_hand_samples: 672
skipped_images: 2181
skipped_boxes: 0
skipped_no_hands: 2177
feature_dim: 126
model_accuracy: 0.9684741488020177
coreml_exported: True
coreml_error: None
dataset_csv: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/phase1_outputs_unified/hand_landmarks_dataset_unified.csv
coreml_model: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/phase1_outputs_unified/hand_gesture_rf_unified.mlmodel
Summary saved to: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/phase1_outputs_unified/training_summary_unified.csv
